In [ ]:
!pip install transformers accelerate datasets torch scikit-learn emoji==0.6.0

In [ ]:
import pandas as pd
import numpy as np
import torch
import csv
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score

data = pd.read_csv('combined_data.csv') #,on_bad_lines='skip',quoting=csv.QUOTE_NONE

X = data['text_without_emoji'].tolist()
y = data['emoji_label'].tolist()

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
# make sure this is 43 emojis
print(f'Total number of unique emoji classes: {len(label_encoder.classes_)}')

X_train, X_test, y_train_enc, y_test_enc = train_test_split(
    X, y_encoded, test_size=0.2, random_state=12, stratify=y_encoded
)

train_df = pd.DataFrame({'text': X_train, 'label': y_train_enc})
test_df = pd.DataFrame({'text': X_test, 'label': y_test_enc})
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

tokenizer = AutoTokenizer.from_pretrained('vinai/bertweet-base', use_fast=True)

def preprocess_text(text):
    new_text = []
    for t in str(text).split(" "):
        t = '@user' if t.startswith('@') and len(t) > 1 else t
        t = 'http' if t.startswith('http') else t
        new_text.append(t)
    return " ".join(new_text)

def tokenize_function(examples):
    processed_texts = [preprocess_text(t) for t in examples['text']]
    return tokenizer(processed_texts, truncation=True, padding='max_length', max_length=64)

tokenized_train_dataset = train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=['text']
)

tokenized_test_dataset = test_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=['text']
)

model = AutoModelForSequenceClassification.from_pretrained(
    'vinai/bertweet-base',
    num_labels=len(label_encoder.classes_),
    hidden_dropout_prob=0.2,
)

training_args = TrainingArguments(
    output_dir='./results/bertweet_replicate',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=16,
    fp16=True,
    learning_rate=2e-5,
    weight_decay=0.01,
    max_grad_norm=1.0,
    optim='adamw_torch',
    logging_steps=50,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    report_to='none'
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    weighted_f1 = f1_score(labels, predictions, average='weighted', zero_division=0)
    accuracy = accuracy_score(labels, predictions)

    return {
        'accuracy': accuracy,
        'weighted_f1': weighted_f1,
    }

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_test_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

results = trainer.evaluate()
print(results)

In [ ]:
from sklearn.metrics import precision_recall_fscore_support
import matplotlib.pyplot as plt
import numpy as np

pred_output = trainer.predict(tokenized_test_dataset)
logits = pred_output.predictions
y_true = pred_output.label_ids
y_pred = np.argmax(logits, axis=-1)
classes = label_encoder.classes_

precisions, recalls, f1s, supports = precision_recall_fscore_support(
    y_true,
    y_pred,
    labels=range(len(classes))
)

df_metrics = pd.DataFrame({
    "emoji": classes,
    "precision": precisions,
    "recall": recalls,
    "f1": f1s,
    "support": supports
})

print(df_metrics)


In [ ]:
x = np.arange(len(classes))

plt.figure(figsize=(12, 6))
plt.bar(x, precisions)
plt.xticks(x, classes, rotation=90)
plt.ylabel("Precision")
plt.title("Precision per Emoji Class")
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 6))
plt.bar(x, recalls)
plt.xticks(x, classes, rotation=90)
plt.ylabel("Recall")
plt.title("Recall per Emoji Class")
plt.tight_layout()
plt.show()
